<a href="https://colab.research.google.com/github/Ragul2526/robot-dataset-quality-checker/blob/main/explore_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install huggingface_hub pandas

In [ ]:
from huggingface_hub import list_repo_files
files = list_repo_files("lerobot/pusht", repo_type="dataset")
print(len(files))
print(files[:25])

In [ ]:
import pandas as pd, json
from huggingface_hub import hf_hub_download
import numpy as np
repo = "lerobot/pusht"

def get(path):
  return hf_hub_download(repo, path, repo_type = "dataset")



In [ ]:
info = json.load(open(get("meta/info.json")))
print({k:v for k,v in info.items() if k!= "features"})
print("\nFeatures")
for name,spec in info['features'].items():
  print(name, spec.get('dtype'),spec.get('shape'))
ep = pd.read_parquet(get('meta/episodes/chunk-000/file-000.parquet'))
print("\nEpisodes: ", ep.shape)
print(ep.columns.tolist())

data = pd.read_parquet(get('data/chunk-000/file-000.parquet'))
print("\nData: ",data.shape)
print(data.head())


In [ ]:
g = data.groupby("episode_index")
summary = pd.DataFrame({
    "frames": g.size(),
    "n_done": g["next.done"].sum(),
    "n_success": g["next.success"].sum(),
    "last_done": g["next.done"].last(),
    "last_success": g["next.success"].last(),
})
print(summary["n_done"].value_counts().head())
print(summary["last_done"].value_counts())
print(summary["last_success"].value_counts())

print("\nBIGGEST JUMPS")
print(data.nlargest(10, "astep")[["episode_index", "frame_index", "ax", "ay", "astep"]])

In [ ]:
import matplotlib.pyplot as plt
last_idx = data.groupby("episode_index")["frame_index"].max()
d = data[data["next.done"]].copy()
d["from_end"] = d["episode_index"].map(last_idx) - d["frame_index"]
print("Done flags, frames from end of episode:\n",d["from_end"].value_counts())

print("\nEpisodes with any success frame:", (summary["n_success"] > 0).sum())
print(summary["n_success"].value_counts().head())
last_reward = data.groupby("episode_index")["next.reward"].last()
print("\nFinal reward per episode:")
print(last_reward.describe())

g = data[data["episode_index"] == 159]
fig, axs = plt.subplots(1, 2, figsize=(11, 3))
axs[0].plot(g["frame_index"], g["ax"], label="action x")
axs[0].plot(g["frame_index"], g["sx"], label="state x")
axs[0].set_title("Episode 159: action vs state (x)")
axs[0].legend()
axs[1].plot(g["frame_index"], g["astep"])
axs[1].set_title("Action step size (px)")
plt.tight_layout()
plt.savefig("episode159_jump.png", dpi=150)
plt.show()


In [ ]:
issues = []
def flag(severity, check, episode, detail):
    issues.append({"severity": severity, "check": check,
                   "episode": episode, "detail": detail})

if len(data) != info["total_frames"]:
    flag("error", "frame_count", "ALL", f"{len(df)} rows vs {info['total_frames']} in info.json")
if (data["index"].diff().dropna() != 1).any():
    flag("error", "global_index", "ALL", "index column is not contiguous")
if sorted(data["episode_index"].unique()) != list(range(info["total_episodes"])):
    flag("error", "episode_ids", "ALL", "episode ids are not 0..N-1")

n_success = int(data["next.success"].sum())
final_reward = data.groupby("episode_index")["next.reward"].last()
if n_success == 0:
    flag("warning", "success_never_true", "ALL",
         f"next.success is never True; final reward max is {final_reward.max():.3f}, "
         f"so it cannot be used to filter demonstrations")

data["astep"] = np.hypot(data.groupby("episode_index")["ax"].diff(),
                       data.groupby("episode_index")["ay"].diff())

for e, g in data.groupby("episode_index"):
    g = g.sort_values("frame_index").reset_index(drop=True)
    n = len(g)

    if n != ep.loc[e, "length"]:
        flag("error", "length_mismatch", e, f"{n} frames vs {ep.loc[e, 'length']} in metadata")
    if g["frame_index"].iloc[0] != 0 or (g["frame_index"].diff().dropna() != 1).any():
        flag("error", "frame_index", e, "frame_index not 0,1,2,...")
    if not np.allclose(g["timestamp"].diff().dropna(), 1 / fps, atol=1e-3):
        flag("error", "timestamp_drift", e, "frame spacing differs from 1/fps")
    if not np.isfinite(g[["sx", "sy", "ax", "ay"]].to_numpy()).all():
        flag("error", "nan_or_inf", e, "non-finite values in state/action")

    done_pos = list(np.flatnonzero(g["next.done"].to_numpy()))
    if done_pos != [n - 2, n - 1]:
        flag("error", "done_flag", e, f"done on frames {done_pos}, expected [{n-2}, {n-1}]")

    if g[["ax", "ay"]].std().max() < 1.0:
        flag("error", "static_action", e, "action barely changes")


    step = g["astep"].to_numpy()
    med = np.nanmedian(step)
    mad = np.nanmedian(np.abs(step - med))
    for t in np.flatnonzero(step > max(med + 10 * mad, 60)):
        t2 = min(t + 5, n - 1)
        moved = np.hypot(g.loc[t2, "sx"] - g.loc[t - 1, "sx"],
                         g.loc[t2, "sy"] - g.loc[t - 1, "sy"])
        ratio = moved / step[t]
        if ratio < 0.3:
            flag("warning", "action_jump_unfollowed", e,
                 f"frame {t}: action jumped {step[t]:.0f}px, state moved only {ratio:.0%} of it")
        else:
            flag("info", "action_jump_followed", e,
                 f"frame {t}: action jumped {step[t]:.0f}px, state followed ({ratio:.0%})")


L = ep["length"]
q1, q3 = L.quantile([0.25, 0.75])
lo, hi = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
for e, n in L.items():
    if n < lo or n > hi:
        flag("info", "length_outlier", e, f"{n} frames (normal range {lo:.0f}-{hi:.0f})")

issues_df = pd.DataFrame(issues)
print(issues_df.groupby(["severity", "check"]).size())
print(issues_df[issues_df["severity"] != "info"])
issues_df.to_csv("issues.csv", index=False)

In [ ]:
max_reward = data.groupby("episode_index")["next.reward"].max()
print("Episodes with max reward >= 0.95:", (max_reward >= 0.95).sum(), "of", len(max_reward))
print(max_reward.describe())

fig, axs = plt.subplots(1, 3, figsize=(15, 3.5))

axs[0].hist(ep["length"], bins=25)
axs[0].axvline(lo, ls="--", c="r"); axs[0].axvline(hi, ls="--", c="r")
axs[0].set_title("Episode lengths (red = IQR bounds)")
axs[0].set_xlabel("frames")

axs[1].hist(max_reward, bins=25)
axs[1].axvline(0.95, ls="--", c="r")
axs[1].set_title("Max reward per episode (red = 0.95 success line)")

axs[2].hist(data["astep"].dropna(), bins=60, log=True)
axs[2].set_title("Action step size (log scale)")
axs[2].set_xlabel("px per frame")

plt.tight_layout()
plt.savefig("summary.png", dpi=150)
plt.show()

print(issues_df.groupby("severity").size())